In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt
from scipy.io import loadmat
from PIL import Image
from transformers import ViTModel, ViTConfig
from tqdm import tqdm
import random
import os
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 设置随机种子为42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 设置随机种子
set_seed(42)

In [ ]:
# Dataset for Images and NIR
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, image_root, nir_path, transform=None):
        self.image_dataset = datasets.ImageFolder(image_root, transform=transform)
        self.nir_data = sio.loadmat(nir_path)['nir_data']  # assuming key is 'nir_data'
        self.labels = sio.loadmat(nir_path)['labels'].squeeze() - 1  # 0-indexed

    def __len__(self):
        return len(self.image_dataset)

    def __getitem__(self, idx):
        image, label = self.image_dataset[idx]
        nir = torch.tensor(self.nir_data[idx], dtype=torch.float32)
        return image, nir, label

# Pyramid Feature Extractor for Images
class PyramidCNN(nn.Module):
    def __init__(self):
        super(PyramidCNN, self).__init__()
        self.base = models.resnet18(pretrained=True)
        self.base.fc = nn.Identity()
        
    def forward(self, x):
        x = self.base(x)
        return x

# Attention-based 1D CNN for NIR
class AttentionCNN1D(nn.Module):
    def __init__(self, input_dim):
        super(AttentionCNN1D, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.attention = nn.Sequential(
            nn.Conv1d(64, 1, kernel_size=1),
            nn.Softmax(dim=-1)
        )
        self.fc = nn.Linear(input_dim, 128)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        attn = self.attention(x)
        x = (x * attn).sum(dim=-1)
        x = self.fc(x)
        return x

# Fusion Model
class MultimodalFusion(nn.Module):
    def __init__(self, img_dim=512, nir_dim=128, num_classes=3):
        super(MultimodalFusion, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(img_dim + nir_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, img_feat, nir_feat):
        x = torch.cat((img_feat, nir_feat), dim=1)
        out = self.fc(x)
        return out

# Student and Teacher Models
class StudentModel(nn.Module):
    def __init__(self):
        super(StudentModel, self).__init__()
        self.img_encoder = PyramidCNN()
        self.nir_encoder = AttentionCNN1D(input_dim=300)
        self.fusion = MultimodalFusion()

    def forward(self, img, nir):
        img_feat = self.img_encoder(img)
        nir_feat = self.nir_encoder(nir)
        output = self.fusion(img_feat, nir_feat)
        return output

class TeacherModel(StudentModel):
    pass  # Same structure but pre-trained better

# Knowledge Distillation Loss
def distillation_loss(student_logits, teacher_logits, true_labels, T=2.0, alpha=0.7):
    loss = nn.KLDivLoss()(F.log_softmax(student_logits/T, dim=1),
                          F.softmax(teacher_logits/T, dim=1)) * (T * T * alpha) + \
           F.cross_entropy(student_logits, true_labels) * (1. - alpha)
    return loss

In [ ]:
# Training

def train_model(student, teacher, dataloader, epochs=20, lr=1e-4):
    optimizer = optim.Adam(student.parameters(), lr=lr)
    teacher.eval()
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for imgs, nirs, labels in dataloader:
            imgs, nirs, labels = imgs.cuda(), nirs.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_logits = student(imgs, nirs)
            with torch.no_grad():
                teacher_logits = teacher(imgs, nirs)
            loss = distillation_loss(student_logits, teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(dataloader):.4f}")

# Evaluation and Visualization

def evaluate_model(model, dataloader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for imgs, nirs, labels in dataloader:
            imgs, nirs = imgs.cuda(), nirs.cuda()
            outputs = model(imgs, nirs)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.show()

    y_true_bin = np.eye(3)[y_true]
    y_prob = np.array(y_prob)
    for i in range(3):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'Class {i} (AUC={roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.show()


In [ ]:
# Main Execution
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset = MultimodalDataset(
        image_root=r"L:\\常惠林\\萎凋\\所有样本",
        nir_path=r"L:\\常惠林\\萎凋\\NIR.mat",
        transform=data_transform
    )
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

    teacher = TeacherModel().to(device)
    student = StudentModel().to(device)

    # Assume teacher is already trained, or load pretrained weights here
    train_model(student, teacher, dataloader, epochs=20)
    evaluate_model(student, dataloader)
